# 03 — Build Gold Reporting Outputs
## Goal
 - read project config
 - read widget parameters
 - read the clean silver sales table
  - apply reusable reporting filters
  - create gold reporting tables
  - print useful run context

In [0]:
import yaml
import sys
from pyspark.sql import functions as F

## Load project config

In [0]:
config_path = "/Workspace/Repos/adb-tetiana@startsteps.org/beiersdorf-live-demo/config/project_config.yml"

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

catalog_name = config["catalog"]
schema_name = config["schema"]

silver_sales_table = f"{catalog_name}.{schema_name}.{config['tables']['silver_sales']}"
gold_market_daily_summary = f"{catalog_name}.{schema_name}.{config['tables']['gold_market_daily_summary']}"
gold_brand_summary = f"{catalog_name}.{schema_name}.{config['tables']['gold_brand_summary']}"
gold_inventory_summary = f"{catalog_name}.{schema_name}.{config['tables']['gold_inventory_summary']}"

print("Silver sales table:", silver_sales_table)
print("Gold market daily summary table:", gold_market_daily_summary)
print("Gold brand summary table:", gold_brand_summary)
print("Gold inventory summary table:", gold_inventory_summary)

## Import reusable sales filtering transformations

In [0]:
src_path = "/Workspace/Repos/adb-tetiana@startsteps.org/beiersdorf-live-demo/src"

In [0]:
if src_path not in sys.path:
    sys.path.append(src_path)

from transformations.sales_filters import (
    filter_report_date_range,
    filter_market,
    filter_brand
    )

In [0]:
print(src_path)

## Recreate widgets so the notebook can also run independently

In [0]:
dbutils.widgets.text(name="report_start_date", defaultValue="2026-04-25", label="Report Start Date")
dbutils.widgets.text(name="report_end_date", defaultValue="2026-04-30", label="Report End Date")
dbutils.widgets.dropdown("market", "ALL", ["ALL", "DE", "FR", "ES", "IT"], "Market")
dbutils.widgets.dropdown("brand", "ALL", ["ALL", "NIVEA", "EUCERIN", "LABELLO", "HANSAPLAST"], "Brand")

report_start_date = dbutils.widgets.get("report_start_date")
report_end_date = dbutils.widgets.get("report_end_date")
market = dbutils.widgets.get("market")
brand = dbutils.widgets.get("brand")

print("Report Start date:", report_start_date)
print("Report end date:", report_end_date)
print("Market:", market)
print("Brand:", brand)

## Read silver sales data

In [0]:
silver_sales_df = spark.table(silver_sales_table)

silver_count = silver_sales_df.count()

print("Silver sales table:", silver_sales_table)
print("Silver row count before reporting filters:", silver_count)

display(silver_sales_df)

## Apply reporting scope filters

In [0]:
reporting_df =(
    silver_sales_df
    .transform(filter_report_date_range(report_start_date, report_end_date))
    .transform(filter_market(market))
    .transform(filter_brand(brand))
)

reporting_count = reporting_df.count()

print("Reporting row count after filters:", reporting_count)
print("Applied report_date filter:", report_start_date, report_end_date)
print("Applied market filter:", market)
print("Applied brand filter:", brand)

display(reporting_df)

## Create temporary view for SQL reporting

In [0]:
reporting_df.createOrReplaceTempView("vw_reporting_sales")

print("Created temp view: vw_reporting_sales")

In [0]:
%sql
SELECT * FROM vw_reporting_sales

## Build gold market daily summary

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {gold_market_daily_summary} AS
    SELECT
     report_day,
      market,
      ROUND(SUM(sales_amount), 2) AS total_sales,
      SUM(units_sold) AS total_units,
      SUM(is_high_value_market_row) AS high_value_rows
    FROM vw_reporting_sales
    GROUP BY report_day, market      
          """)

print("Built table:", gold_market_daily_summary)

In [0]:
display(spark.table(gold_market_daily_summary))

## Build gold brand summary

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {gold_brand_summary} AS
SELECT
    report_day,
    brand,
    ROUND(SUM(sales_amount), 2) AS total_sales,
    SUM(units_sold) AS total_units
FROM vw_reporting_sales
GROUP BY report_day, brand
""")

print("Built table:", gold_brand_summary)

In [0]:
display(spark.table(gold_brand_summary))

## Build gold inventory summary

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {gold_inventory_summary} AS
SELECT
    report_day,
    market,
    brand,
    ROUND(AVG(inventory_level), 2) AS avg_inventory_level
FROM vw_reporting_sales
GROUP BY report_day, market, brand
""")

print("Built table:", gold_inventory_summary)

In [0]:
display(spark.table(gold_inventory_summary))